# 00 · Entorno y corpus

## Propósito

Confirmar dos cosas antes de empezar: que el entorno funciona, y que el corpus
es lo que dice ser.

No vamos a construir nada acá. El corpus ya está armado, en `datos/corpus.jsonl`.
Lo que vamos a hacer es **auditarlo**: abrir los datos primarios, mirar qué son
realmente, y verificar que los números de `datos/FUENTE.md` coincidan con lo que
hay en el archivo.

Esto es lo que le pasa a cualquiera que recibe datos de otra persona. Nadie te
va a entregar un corpus limpio con una explicación completa. Y "lo descargué y
lo usé" es la forma más rápida de publicar un error ajeno con tu nombre.

## Qué consume

- `datos/blog/*.txt` — los datos primarios
- `datos/corpus.jsonl` — el corpus reconstruido
- `datos/FUENTE.md` — la documentación que vamos a verificar

## 1. ¿Está el entorno listo?

In [ ]:
# sys: para preguntarle a Python qué versión de sí mismo está corriendo.
import sys

# spacy: la librería de análisis lingüístico de toda la unidad.
import spacy

# sys.version trae la versión y más datos. split() la corta en piezas: la primera es el número.
partes_de_la_version = sys.version.split()
version_de_python = partes_de_la_version[0]

print("Python:", version_de_python)
# __version__: la versión instalada de la librería
print("spaCy :", spacy.__version__)

Si acá falla el `import spacy`, no estás usando el intérprete del entorno
virtual. En VS Code: `Ctrl+Shift+P` → *Python: Select Interpreter* → elegí `.venv`.

In [ ]:
# Cargamos el modelo de español.
# es_core_news_lg  ->  es: español
#                      core: pipeline completo
#                      news: entrenado con textos periodísticos
#                      lg: tamaño grande (631 MB)
nlp = spacy.load("es_core_news_lg")

# meta: un diccionario con los datos del modelo (nombre, versión, idioma...)
print("modelo :", nlp.meta["name"], nlp.meta["version"])
print("idioma :", nlp.meta["lang"])

Ese `news` del nombre no es un detalle decorativo: dice **con qué textos
aprendió** el modelo. Nuestro corpus son entradas de blog rioplatense de 2004.
No es lo mismo, y en el cuaderno 04 vamos a medir cuánto importa esa diferencia.

## 2. Los archivos de `blog/` no son texto

La carpeta `datos/blog/` tiene doce archivos con extensión `.txt`. Abrí uno con
un editor de texto antes de correr la celda siguiente. Vas a ver caracteres rotos.

Miremos por qué.

In [ ]:
# pathlib: para armar rutas que funcionen igual en Windows, macOS y Linux.
from pathlib import Path

ruta_2004 = Path("datos/blog/2004.txt")

# Abrimos en modo binario ("rb") para ver los bytes crudos, sin interpretarlos.
archivo = open(ruta_2004, "rb")
primeros_bytes = archivo.read(12)  # read(12): lee los primeros 12 bytes
archivo.close()  # close(): cierra el archivo cuando terminamos de leer

print(primeros_bytes)

Ese `\x80\x03X` del principio es la firma de un objeto **pickle**: el formato
en que Python guarda objetos en disco. No es texto. Por eso el editor muestra
basura.

Para leerlo hay que usar la librería correspondiente.

In [ ]:
# pickle: para leer objetos de Python guardados en disco.
import pickle

archivo = open(ruta_2004, "rb")
texto_2004 = pickle.load(archivo)  # pickle.load(): reconstruye el objeto guardado
archivo.close()

# type(): qué clase de objeto es. __name__: el nombre de esa clase, como texto
tipo_de_objeto = type(texto_2004)
nombre_del_tipo = tipo_de_objeto.__name__
cantidad_caracteres = len(texto_2004)
# "\n" es el carácter de salto de línea. count() cuenta cuántas veces aparece.
cantidad_saltos = texto_2004.count("\n")

print("tipo       :", nombre_del_tipo)
# :, separa los miles con una coma
print("caracteres :", f"{cantidad_caracteres:,}")
print("saltos de línea:", cantidad_saltos)

**Un solo salto de línea en 190.786 caracteres.** Todos los textos de 2004
están pegados en un único string, sin párrafos, sin títulos y sin fechas.

Miremos el principio.

In [ ]:
# [:500]: los primeros 500 caracteres del texto
print(texto_2004[:500])

Fijate en algo: `...detalles con el lector.La idea nació...`

Un punto pegado a una mayúscula, sin espacio. **Ahí terminaba un párrafo** y el
salto de línea se borró al concatenar. Esa marca es lo que permitió reconstruir
el corpus, y está explicada en `datos/FUENTE.md`.

Abrí ese archivo ahora y leelo entero. Son cinco minutos.

## 3. Verificar lo que dice `FUENTE.md`

`FUENTE.md` afirma varias cosas. No hay razón para creerle. Comprobémoslas.

In [ ]:
# pandas: para trabajar el corpus como una tabla.
import pandas as pd

# read_json con lines=True lee un archivo .jsonl: un objeto JSON por línea.
corpus = pd.read_json("datos/corpus.jsonl", lines=True)

cantidad_parrafos = len(corpus)
print("párrafos:", f"{cantidad_parrafos:,}")
print()
# display(): muestra la tabla con formato. head(3): solo las primeras 3 filas
display(corpus.head(3))

if cantidad_parrafos != 5428:
    print("ATENCIÓN: el corpus debería tener 5.428 párrafos.")
    print("Si el número no coincide, volvé a generarlo con datos/reconstruir_corpus.py.")

**Afirmación 1:** el corpus tiene 5.428 párrafos.

**Afirmación 2:** conserva el 100% de los caracteres de `blog/`.

In [ ]:
# Sumamos los caracteres del corpus reconstruido.
# sum(): suma todos los valores de la columna
caracteres_corpus = corpus["caracteres"].sum()

# Buscamos los doce archivos originales.
carpeta_blog = Path("datos/blog")
archivos_del_blog = carpeta_blog.glob("*.txt")  # glob("*.txt"): los archivos que terminan en .txt
archivos_ordenados = sorted(archivos_del_blog)  # sorted(): en orden, de 2004 a 2015

# Sumamos los caracteres de los doce archivos originales.
caracteres_original = 0
for ruta in archivos_ordenados:
    archivo = open(ruta, "rb")
    texto = pickle.load(archivo)
    archivo.close()
    caracteres_original = caracteres_original + len(texto)

porcentaje_conservado = 100 * caracteres_corpus / caracteres_original

print("caracteres en blog/       :", f"{caracteres_original:,}")
print("caracteres en corpus.jsonl:", f"{caracteres_corpus:,}")
# :.2f: con dos decimales
print("conservado                :", f"{porcentaje_conservado:.2f}%")

La diferencia son espacios en blanco al principio y al final de cada párrafo.

**Afirmación 3:** la cantidad de párrafos por año.

In [ ]:
# groupby(): junta las filas que tienen el mismo año. size(): cuántas filas hay en cada grupo
por_anio = corpus.groupby(by="anio").size()
print(por_anio)

## 4. Lo que este corpus no puede contestar

Esto es lo más importante del cuaderno.

El scraping original bajó **386 textos por separado** y los concatenó por año
antes de guardarlos. Los límites entre un texto y el siguiente se perdieron, y
la fuente ya no está en línea: no se pueden volver a bajar.

Entonces:

- La unidad de análisis es el **párrafo**, no el texto.
- No se puede comparar un texto con otro.
- No se puede contar en cuántos textos aparece algo.
- 2008 tiene 813 párrafos y 2009 tiene 116, pero eso **no** mide productividad:
  no sabemos cuántos textos hubo ni de qué largo.

Si tu pregunta necesita el texto como unidad, este corpus no sirve y hay que
buscar otro. En el trabajo práctico vas a tener otras opciones.

### Y hay otro límite, que no es un daño

Todo lo anterior es lo que se **perdió** en el camino: un archivo roto que nadie puede
reparar. Pero hay un límite anterior a eso, y no vino de ningún error. Es el recorte
mismo del corpus.

Acá adentro hay **una sola voz**.

- **Un autor.** Hernán Casciari, nacido en Mercedes, provincia de Buenos Aires, en 1971.
  No es una muestra de la escritura argentina: es una persona.
- **Un registro.** Entradas de blog. Primera persona, tono conversacional, textos cortos
  pensados para leerse en pantalla. No hay prosa académica, ni burocrática, ni oral
  transcripta, ni periodística formal.
- **Un lugar que no es el que parece.** Casciari se radicó en Barcelona en el año 2000.
  Los doce años de este corpus están escritos **desde España**, para lectores repartidos
  entre España, la Argentina y el resto de Latinoamérica. Es castellano rioplatense
  escrito lejos del Río de la Plata, y para un público que no es solo de acá.
- **Una década, y un final sin explicar.** El corpus arranca en 2004 y termina en 2015.
  La captura se hizo en 2019. O sea que hay cuatro años entre el último texto y el
  scraping, y `FUENTE.md` no dice por qué. **No sabemos si el blog paró, si el scraper
  cortó ahí, o si esos años se perdieron.** Un corpus siempre empieza y termina en
  alguna parte, y casi nunca es por una razón metodológica.

### Entonces, qué se puede afirmar

Con este corpus podés decir cosas sobre **cómo escribe Casciari**, y compararlo consigo
mismo a lo largo de doce años. Eso es bastante: alcanza para todo lo que vamos a hacer
en la unidad.

Lo que no podés decir es nada que empiece con *«los argentinos»*, *«la gente»* o *«los
blogs»*. Cada vez que te tiente una frase así, acordáte de que tenés una sola voz para
sostenerla.

Esto no es una advertencia moral: es una restricción técnica. Una afirmación sobre una
población necesita una muestra de esa población, y acá la muestra tiene tamaño uno.

> **Y no se arregla cambiando de corpus.** En el `08` vas a elegir otro material. El
> problema no desaparece: cambia de forma. Ahí la pregunta va a ser cuál es **tu**
> recorte y qué deja afuera.

> ### Perilla
>
> Cada párrafo trae el campo `caracteres`. El más corto tiene 5 caracteres y el
> más largo 11.237.
>
> Mirá los extremos con la celda de abajo. Los cortos son líneas de diálogo
> (`ALEX.- Igual`). El más largo casi seguro no es un párrafo: es un tramo donde
> la marca de corte no apareció.
>
> **La pregunta es tuya:** ¿son ruido o son datos? No hay respuesta general.
> Depende de qué vayas a preguntar. Lo que no se puede es filtrarlos sin decirlo.

In [ ]:
# Para cada fila, ¿tiene menos de 40 caracteres? Da una columna de True y False.
es_corto = corpus["caracteres"] < 40
# Nos quedamos solo con las filas que dieron True.
cortos = corpus[es_corto]
cantidad_de_cortos = len(cortos)

print("párrafos de menos de 40 caracteres:", cantidad_de_cortos)
print()
primeros_ocho = cortos["texto"].head(8)
for texto in primeros_ocho:
    print(" ", texto)

In [ ]:
# sort_values(): ordena las filas. ascending=False: de mayor a menor
ordenado_por_largo = corpus.sort_values(by="caracteres", ascending=False)
# iloc[0]: la fila que está en la posición 0, o sea la primera
mas_largo = ordenado_por_largo.iloc[0]

print("año:", mas_largo["anio"], "| caracteres:", mas_largo["caracteres"])
print()
comienzo_del_texto = mas_largo["texto"][:400]  # [:400]: los primeros 400 caracteres
print(comienzo_del_texto, "...")

## Lo que hiciste hoy

- Confirmaste que el entorno y el modelo cargan.
- Descubriste que los archivos `.txt` de `blog/` no son texto sino objetos pickle.
- Verificaste con tus propios números las tres afirmaciones de `FUENTE.md`.
- Sabés qué preguntas **no** se le pueden hacer a este corpus.

## Lo que viene

En el `01` vamos a ver qué le agrega spaCy a un texto que Python solo no puede
ver, y por qué `texto.split()` no alcanza.